In [100]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import NoSuchElementException, TimeoutException, ElementNotInteractableException, ElementClickInterceptedException   
from datetime import datetime

from bs4 import BeautifulSoup
import pandas as pd
import re
import numpy as np
import requests
from time import sleep
import random
import urllib.parse

---
### Explorer crawler

---

In [101]:
# Explorer Functions
def time_it(func):
    def wrapper(*args, **kwargs):
        start_time = datetime.now()
        print(f'{func.__name__} == {start_time}')
        result = func(*args, **kwargs)
        elapsed_time = datetime.now() - start_time
        print(f"    {func.__name__} == took: {elapsed_time}")
        return result
    return wrapper


def print_update(func):
    def wrapper(*args, **kwargs):
        print(f'{func.__name__}')
        return func(*args, **kwargs)  # Call the decorated function
    return wrapper


@print_update
def setup_driver():
    options = webdriver.ChromeOptions()
    prefs = {"profile.managed_default_content_settings.images": 2}
    options.add_experimental_option("prefs", prefs)
    options.add_argument("--headless")
    options.add_argument("--window-size=1920,1080")
    driver = webdriver.Chrome(options=options)
    driver.maximize_window()

    driver.get('https://es.wallapop.com/app/search?filters_source=category_navigation&latitude=40.41956&longitude=-3.69196')

    # rejects cookie
    WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.ID, 'onetrust-reject-all-handler')))
    driver.find_element(By.ID, 'onetrust-reject-all-handler').click()

    # clicks tutorial skip button
    WebDriverWait(driver, 5).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip')))
    driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
    driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()
    driver.find_element(By.CSS_SELECTOR, 'walla-button.TooltipWrapper__skip').click()

    return driver


@print_update
def get_explorer_urls(product_id=None, category=24200):
    # gets the corresponding urls to wallapop listing explorer
    url = 'https://es.wallapop.com/app/search?'

    url = url if (category == 0) else url + f'category_ids={category}&'
    url = url if (product_id == None) else url + f'keywords={urllib.parse.quote(product_id)}&'

    url += 'filters_source=quick_filters&latitude=40.41956&longitude=-3.69196&order_by=newest&'
    urls = split_explorer_url(url, product_id, category)

    return urls


@print_update
def split_explorer_url(url, product_id, category):
    # breaks up url into smaller subsets to divide up scrape load
    urls = []

    if product_id:
        # IGNORE write later after figuring out product_id classification module
        pass
    else:
        if category == 24200:
            median, std, step_size = 55, 345, 0.1
            loops = round((median + std)/step_size)

            for i in range(loops):
                min = (step_size * i)
                max = (step_size * (i+1))
                if i == 0:
                    pricequery = f'&max_sale_price={max}'
                elif i + 1 == len(range(loops)):
                    pricequery = f'min_sale_price={min+.01}'
                else:
                    pricequery = f'min_sale_price={min+.01}&max_sale_price={max}'

                urls.append({
                    'explorer_url': url + pricequery,
                    'product_id': category,
                    'date_scraped': np.nan,
                    'scrape_empty': np.nan,
                    'redundant_url': np.nan,
                })
        pass
    return urls


@print_update
def load_more(driver, retries=1):
    # Clicks button that initiates infinite scroll dynamic listings
    try:
        WebDriverWait(driver, retries).until(EC.element_to_be_clickable((By.ID, 'btn-load-more')))
        driver.find_element(By.ID, 'btn-load-more').click()
    except (ElementClickInterceptedException, TimeoutException, ElementNotInteractableException) as e:
        print(f'    Load_more button == {e}')
        retries += 1
        if retries < 5:
            load_more(driver, retries)


@time_it
def crawl_explorer():
    driver = setup_driver()

    try:
        urls = pd.read_csv('data/urls.csv')
    except FileNotFoundError:
        urls = pd.DataFrame(get_explorer_urls())
    urls['date_scraped'] = pd.to_datetime(urls['date_scraped'], errors='coerce')

    current_week = datetime.now().isocalendar()[:2] # isocalendar() returns -> year, week, weekday
    pending_urls = urls[urls['date_scraped'].apply(lambda x: x.isocalendar()[:2] if pd.notnull(x) else None) != current_week]

    listings = pd.DataFrame(columns=['title','href','product_id','listing_id','num_images','img_scr','date_scraped','price','featured','shipping','reserved','walla_pro'])

    for i, v in pending_urls['explorer_url'].items():
        driver.get(v)
        WebDriverWait(driver, 10).until(EC.element_to_be_clickable((By.CSS_SELECTOR, 'a.ItemCardList__item')))
        load_more(driver)

        scraped = []
        scroll_explorer(driver, i)

        # input()
        scraped += scrape_explorer(driver)

        urls.at[i, 'date_scraped'] = datetime.now()
        if scraped:
            urls.at[i, 'scrape_empty'] = False
            temp = pd.DataFrame(scraped)
            if temp.isin(listings.to_dict(orient='list')).all().all():
                urls.at[i, 'redundant_url'] = True
            else:
                urls.at[i, 'redundant_url'] = False
                listings = pd.concat([listings, temp]).drop_duplicates()
                listings.to_csv(f'data/listings/{urls.iloc[i]['product_id']}_{i}_listings.csv', index=False)
            urls.to_csv('data/urls.csv', index=False)
        else: 
            urls.at[i, 'scrape_empty'] = True
            urls.to_csv('data/urls.csv', index=False)


@print_update
def scroll_explorer(driver, i):
    previous_height = driver.execute_script("return document.body.scrollHeight")
    sleep_time = 0.1
    count = 0
    timer = datetime.now()

    # scrolls to the bottom of scroller page until no more listings are loaded on scroll
    while True:
        count +=1
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        sleep(sleep_time)
        new_height = driver.execute_script("return document.body.scrollHeight")

        if new_height == previous_height:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            driver.execute_script("window.scrollBy(0, 99999);")
            sleep_time += sleep_time
            if sleep_time > 25:
                return True
            sleep(sleep_time)
        else:
            sleep_time /= 2

        if count == 50:
            driver.execute_script("window.scrollTo(0, 0);")
            if clear_explorer(driver, i, timer):
                return True
        else:
            previous_height = new_height


@print_update
def check_scrape(driver, i, count=0):
    # checks if num of listings scraped corresponds to number of listings displayed in selenium
    if count > 1:
        print("    Element count doesn't match scrape count after 3 loops")
        return current_scrape
    else:
        elem_count = len(driver.find_elements(By.CSS_SELECTOR, 'a.ItemCardList__item'))
        current_scrape = scrape_explorer(driver)
        if len(current_scrape) == elem_count:
            pass
        else:
            print(f'    Scrape missing - URL {i}')
            print(f'    Element count > scrape: {elem_count > len(current_scrape)}')
            count += 1
            current_scrape = check_scrape(driver, i, count=count)

        return current_scrape


@print_update
def clear_explorer(driver, i, timer):
    global scraped
    sleep(1)
    current_scrape = check_scrape(driver, i)

    temp = pd.DataFrame(current_scrape)
    if temp.isin(pd.DataFrame(scraped).to_dict(orient='list')).all().all():
        print('    Redundant scroll results')
    else:
        scraped += current_scrape
    
    # Find elements and delete all siblings
    element = driver.find_element(By.CSS_SELECTOR, 'a.ItemCardList__item')
    parent = element.find_element(By.XPATH, '..')
    siblings = parent.find_elements(By.XPATH, '*')
    for sibling in siblings:
        driver.execute_script("arguments[0].remove();", sibling)

    elements = driver.find_elements(By.CSS_SELECTOR, 'a.ItemCardList__item')
    for element in elements:
        driver.execute_script("arguments[0].remove();", element)

    # Print updates
    print(f'    current scrape = {elem_count}')
    timespent = datetime.now() - timer
    print(f'    scrape time = {timespent}')
    print(f'    efficiency = {timespent/elem_count}')
    print(f'    total scraped = {len(scraped)}')

    if scroll_explorer(driver, i):
        return True


@print_update
def scrape_explorer(driver):
    listings = []
    cards = driver.find_elements(By.CSS_SELECTOR, 'a.ItemCardList__item')
    for card in cards:
        title = card.get_attribute('title')
        href = card.get_attribute('href')
        listing_id = href.split('-')[-1]
        num_images = len(card.find_elements(By.TAG_NAME, 'li'))
        img_scr = card.find_element(By.TAG_NAME, 'img').get_attribute('src')
        date_scraped = datetime.now()#.strftime('%d-%m-%y')

        price = card.find_element(By.CSS_SELECTOR, 'span.ItemCard__price').text.strip()
        price = float(re.sub(r'[^\d,]', '', price).replace(',', '.'))
        
        try:
            if card.find_element(By.CSS_SELECTOR, 'p.ItemCard__highlight-text.pb-2'):
                featured = True
            else:
                featured = False
        except NoSuchElementException:
            featured = False

        shadow_hosts = card.find_elements(By.CSS_SELECTOR, 'wallapop-badge')
        shipping, reserved, walla_pro = 0, False, False
        for shadow_host in shadow_hosts:
            shadow_root = driver.execute_script('return arguments[0].shadowRoot', shadow_host)
            span = shadow_root.find_element(By.CSS_SELECTOR, 'span').text.strip() 
            if span == 'Sólo venta en persona':
                shipping = 0
            elif span == 'Envío disponible':
                shipping = 1
            elif span == 'Envío gratis':
                shipping = 2
            elif span == 'Reservado':
                reserved = True
            elif span == 'Reacondicionado':
                walla_pro = True

        # product_id missing
        # days_posted missing

        listings.append({
            'title': title,
            'href': href,
            'listing_id': listing_id,
            'num_images': num_images,
            'img_scr': img_scr,
            'date_last_seen': date_scraped,
            'price': price,
            'featured': featured,
            'shipping' : shipping,
            'reserved' : reserved,
            'walla_pro' : walla_pro,
        })
        
    return listings

In [102]:
scraped = []

crawl_explorer()

crawl_explorer == 2024-08-03 12:07:39.626480

setup_driver

get_explorer_urls

split_explorer_url

load_more
scrape_explorer == 2024-08-03 12:07:50.164392
    scrape_explorer == took: 0:00:23.421152
    current scrape = 745
    efficiency = 0:00:00.044711
    total scraped = 745
scrape_explorer == 2024-08-03 12:08:23.901111
    scrape_explorer == took: 0:00:25.210178
    current scrape = 781
    efficiency = 0:00:00.046498
    total scraped = 1526
scrape_explorer == 2024-08-03 12:08:59.810778
    scrape_explorer == took: 0:00:22.276045
    current scrape = 721
    efficiency = 0:00:00.045184
    total scraped = 2247
scrape_explorer == 2024-08-03 12:09:32.855862
    scrape_explorer == took: 0:00:22.152444
    current scrape = 698
    efficiency = 0:00:00.046666
    total scraped = 2945
scrape_explorer == 2024-08-03 12:10:05.753746
    scrape_explorer == took: 0:00:23.663142
    current scrape = 759
    efficiency = 0:00:00.046971
    total scraped = 3704
scrape_explorer == 2024-08-03 12

WebDriverException: Message: unknown error: session deleted because of page crash
from unknown error: cannot determine loading status
from tab crashed
  (Session info: chrome-headless-shell=127.0.6533.89)
Stacktrace:
0   chromedriver                        0x0000000102499088 cxxbridge1$str$ptr + 1887276
1   chromedriver                        0x0000000102491764 cxxbridge1$str$ptr + 1856264
2   chromedriver                        0x00000001020a0694 cxxbridge1$string$len + 88116
3   chromedriver                        0x000000010208bb08 cxxbridge1$string$len + 3240
4   chromedriver                        0x0000000102089954 core::str::slice_error_fail::he7b2aa4898bc357e + 59976
5   chromedriver                        0x000000010208a5c0 core::str::slice_error_fail::he7b2aa4898bc357e + 63156
6   chromedriver                        0x0000000102097e98 cxxbridge1$string$len + 53304
7   chromedriver                        0x000000010211c30c cxxbridge1$string$len + 595116
8   chromedriver       

#### Category_ids
Motor y accesorios = 12800
Moda y accesorios = 12465
Inmobiliaria = 200
Tecnología y electrónica = 24200
Deporte y ocio = 12579
Bicicletas = 17000
Hogar y jardín = 12467
Electrodomésticos = 13100
Cine, libros y música = 12463
Niños y bebés = 12461
Coleccionismo = 18000
Construcción y reformas = 19000
Industria y agricultura = 20000
Otros = 12485

---
### Parse
---

In [ ]:
driver = setup_driver()
url = 'https://es.wallapop.com/app/search?filters_source=category_navigation&latitude=40.41956&longitude=-3.69196'
driver.get(url)

scroll_explorer(driver)

listings = []
cards = driver.find_elements(By.CSS_SELECTOR, 'a.ItemCardList__item')

for card in cards:
    title = card.get_attribute('title')
    href = card.get_attribute('href')
    listing_id = href.split('-')[-1]
    num_images = len(card.find_elements(By.TAG_NAME, 'li'))
    img_scr = card.find_element(By.TAG_NAME, 'img').get_attribute('src')
    date_scraped = datetime.now()
    date_last_seen = datetime.now()

    price = card.find_element(By.CSS_SELECTOR, 'span.ItemCard__price').text.strip()
    price = float(re.sub(r'[^\d,]', '', price).replace(',', '.'))
    
    try:
        if card.find_element(By.CSS_SELECTOR, 'p.ItemCard__highlight-text.pb-2'):
            featured = True
        else:
            featured = False
    except NoSuchElementException:
        featured = False

    shipping, reserved, walla_pro = 0, False, False
    shadow_hosts = card.find_elements(By.CSS_SELECTOR, 'wallapop-badge')
    for shadow_host in shadow_hosts:
        shadow_root = driver.execute_script('return arguments[0].shadowRoot', shadow_host)
        span = shadow_root.find_element(By.CSS_SELECTOR, 'span').text
        if span == 'Sólo venta en persona':
            shipping = 0
        elif span == 'Envío disponible':
            shipping = 1
        elif span == 'Envío gratis':
            shipping = 2
        elif span == 'Reservado':
            reserved = True
        elif span == 'Reacondicionado':
            walla_pro = True

    # product_id missing
    # days_posted missing

    listings.append({
        'title': title,
        'href': href,
        'listing_id': listing_id,
        'num_images': num_images,
        'img_scr': img_scr,
        'date_scraped': date_scraped,
        'date_last_seen': date_last_seen,
        'price': price,
        'featured': featured,
        'shipping' : shipping,
        'reserved' : reserved,
        'walla_pro' : walla_pro,
    })

listings

---
### Product clasifier
---

In [ ]:
'''
It sounds like you're on the right track with normalizing the strings! Here are some additional steps you can take to further categorize and group similar book titles:

1. **Spell Correction**: Use a spell-checking library like `pyspellchecker` or `SymSpell` to correct common typos and misspellings.

2. **Remove Special Characters**: Strip out any special characters, punctuation, and numbers that are not part of the book title.

3. **Tokenization**: Break down the titles into individual words (tokens) and remove common stop words (like "and", "the", etc.).

4. **Stemming/Lemmatization**: Reduce words to their base or root form using libraries like `nltk` or `spaCy`.

5. **Fuzzy Matching**: Use fuzzy matching techniques (e.g., `fuzzywuzzy` library) to compare and group titles that are similar but not identical.

6. **Clustering**: Apply clustering algorithms (e.g., K-means, DBSCAN) to group similar titles together based on their tokenized and normalized forms.

7. **Manual Review**: After automated processing, a manual review might still be necessary to ensure accuracy, especially for edge cases.

Here's a sample Python code snippet to get you started with some of these steps:
'''


import re
from spellchecker import SpellChecker
from fuzzywuzzy import fuzz
from fuzzywuzzy import process

# Sample list of book titles
titles = ["HP and the sorcerer's stone", "Harry Potter and the Sorcerer's Stone", "Harry Pótter and the Sorcerer's Stone", ...]

# Normalize titles
def normalize_title(title):
    title = title.lower().strip()
    title = re.sub(r'[^\w\s]', '', title)  # Remove special characters
    title = re.sub(r'\d+', '', title)  # Remove numbers
    return title

# Spell correction
spell = SpellChecker()
def correct_spelling(title):
    corrected_title = " ".join([spell.correction(word) for word in title.split()])
    return corrected_title

# Apply normalization and spell correction
normalized_titles = [normalize_title(title) for title in titles]
corrected_titles = [correct_spelling(title) for title in normalized_titles]

# Fuzzy matching
def group_titles(titles):
    grouped_titles = {}
    for title in titles:
        match = process.extractOne(title, grouped_titles.keys(), scorer=fuzz.token_sort_ratio)
        if match and match[1] > 80:  # Adjust threshold as needed
            grouped_titles[match[0]].append(title)
        else:
            grouped_titles[title] = [title]
    return grouped_titles

grouped_titles = group_titles(corrected_titles)

# Print grouped titles
for key, group in grouped_titles.items():
    print(f"Group: {key}")
    for title in group:
        print(f" - {title}")
